# 📧 Day 03a: Email Protocols — SMTP, POP3, IMAP

---

## 🎯 What You'll Master Today
- How email works end-to-end
- SMTP for sending, POP3/IMAP for receiving
- Port numbers and security variants
- MIME, SPF, DKIM, DMARC

---

```
╔══════════════════════════════════════════════════════════════════════╗
║                  HOW EMAIL WORKS                                     ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  alice@gmail.com → bob@yahoo.com                                    ║
║                                                                      ║
║  ┌────────┐  SMTP   ┌──────────┐  SMTP   ┌──────────┐  POP3/IMAP  ║
║  │ Alice  │ ──────→ │  Gmail   │ ──────→ │  Yahoo   │ ──────────→ ║
║  │ Client │         │  MTA     │         │  MTA     │             ║
║  └────────┘         └──────────┘         └──────────┘             ║
║    (MUA)              (relay)              (delivery)    ┌────────┐║
║                                                          │  Bob   │║
║  MUA = Mail User Agent (Outlook, Gmail app)             │ Client │║
║  MTA = Mail Transfer Agent (Postfix, Sendmail)          └────────┘║
║  MDA = Mail Delivery Agent (stores in mailbox)                    ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. SMTP Session Simulation
# ============================================

def simulate_smtp():
    """Simulate SMTP conversation (RFC 5321)."""
    conversation = [
        ("S", "220 smtp.gmail.com ESMTP ready"),
        ("C", "EHLO client.example.com"),
        ("S", "250-smtp.gmail.com Hello\n250-SIZE 35882577\n250-8BITMIME\n250-STARTTLS\n250 OK"),
        ("C", "STARTTLS"),
        ("S", "220 Ready for TLS"),
        ("C", "  [TLS handshake happens here]"),
        ("C", "AUTH LOGIN"),
        ("S", "334 VXNlcm5hbWU6"),
        ("C", "  [base64-encoded credentials]"),
        ("S", "235 Authentication successful"),
        ("C", "MAIL FROM:<alice@gmail.com>"),
        ("S", "250 OK"),
        ("C", "RCPT TO:<bob@yahoo.com>"),
        ("S", "250 OK"),
        ("C", "DATA"),
        ("S", "354 Start mail input; end with <CRLF>.<CRLF>"),
        ("C", "From: alice@gmail.com\nTo: bob@yahoo.com\nSubject: Hello\n\nHi Bob!\n."),
        ("S", "250 OK: queued as ABC123"),
        ("C", "QUIT"),
        ("S", "221 Bye"),
    ]
    
    print("  ═══ SMTP Session (Port 587 with STARTTLS) ═══\n")
    for role, msg in conversation:
        prefix = "  Server:" if role == "S" else "  Client:"
        for line in msg.split("\n"):
            print(f"    {prefix} {line}")
        prefix = "        "

simulate_smtp()

print("\n  SMTP Ports:")
print("    25:  Server-to-server relay (often blocked by ISPs)")
print("    587: Client submission (STARTTLS — recommended)")
print("    465: Client submission (implicit SSL — deprecated, revived)")

In [ ]:
# ============================================
# 2. POP3 vs IMAP Comparison
# ============================================

print("""
  ┌───────────────────┬─────────────────────┬─────────────────────────┐
  │ Feature            │ POP3                │ IMAP                    │
  ├───────────────────┼─────────────────────┼─────────────────────────┤
  │ Full name          │ Post Office Protocol│ Internet Message Access  │
  │ Port (plain)       │ 110                 │ 143                     │
  │ Port (SSL)         │ 995                 │ 993                     │
  │ Storage            │ Downloads to client │ Keeps on server         │
  │ Multi-device       │ ❌ (1 device)       │ ✅ (sync across all)    │
  │ Offline access     │ ✅ (downloaded)     │ ⚠️ (need to sync)      │
  │ Server storage     │ Low (deleted after) │ High (all mail stored)  │
  │ Folders/Labels     │ ❌ (inbox only)     │ ✅ (server-side folders)│
  │ Search on server   │ ❌                  │ ✅                       │
  │ Bandwidth          │ High (full download)│ Low (headers first)     │
  │ Use case           │ Single device, low  │ Multi-device, modern    │
  │                    │ storage server      │ email (Gmail, Outlook)  │
  └───────────────────┴─────────────────────┴─────────────────────────┘

  Modern world: IMAP dominates. POP3 still used for:
    - Backup (download everything, delete from server)
    - Low-storage email servers
    - Privacy (no server-side copy)
""")

In [ ]:
# ============================================
# 3. Email Security: SPF, DKIM, DMARC
# ============================================

print("""
  ═══ EMAIL SECURITY (Anti-Spoofing) ═══

  Problem: SMTP has NO built-in authentication.
  Anyone can send email claiming to be anyone!

  Solution: 3 DNS-based verification mechanisms:

  1. SPF (Sender Policy Framework)
     ┌─────────────────────────────────────────────────────┐
     │ DNS TXT record: "v=spf1 include:_spf.google.com ~all"
     │ Meaning: Only Google's IPs can send for this domain │
     │ Checks: Is the sending IP authorized?               │
     └─────────────────────────────────────────────────────┘

  2. DKIM (DomainKeys Identified Mail)
     ┌─────────────────────────────────────────────────────┐
     │ Sending server signs email with private key         │
     │ Public key published in DNS TXT record              │
     │ Receiver verifies signature → email not tampered    │
     └─────────────────────────────────────────────────────┘

  3. DMARC (Domain Message Auth, Reporting & Conformance)
     ┌─────────────────────────────────────────────────────┐
     │ Policy: what to do if SPF/DKIM fail                 │
     │ p=none (monitor), p=quarantine (spam), p=reject     │
     │ Also sends reports back to domain owner             │
     └─────────────────────────────────────────────────────┘

  All three together = strong email authentication.
  Gmail/Outlook require these for bulk senders (2024+).
""")

In [ ]:
# ============================================
# 4. All Email Ports Summary
# ============================================

print("""
  ╔══════════════════════════════════════════════════════════╗
  ║               EMAIL PORT NUMBERS                        ║
  ╠══════════════════════════════════════════════════════════╣
  ║  Protocol │ Plain │ SSL/TLS │ STARTTLS │ Direction      ║
  ║  ─────────┼───────┼─────────┼──────────┼──────────────  ║
  ║  SMTP     │  25   │  465    │   587    │ Sending        ║
  ║  POP3     │  110  │  995    │   110    │ Receiving      ║
  ║  IMAP     │  143  │  993    │   143    │ Receiving      ║
  ╚══════════════════════════════════════════════════════════╝

  Best practices:
  - Use port 587 (STARTTLS) for sending — NOT port 25
  - Use IMAPS (993) for receiving — always encrypted
  - Never use plain-text ports in production
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | SMTP vs POP3 vs IMAP? | SMTP=sending. POP3=download+delete. IMAP=sync, keep on server |
| 2 | POP3 vs IMAP? | POP3: single device, downloads. IMAP: multi-device, server-side folders |
| 3 | What is STARTTLS? | Upgrades plain-text connection to encrypted (port 587 for SMTP) |
| 4 | What is SPF? | DNS record listing authorized sending IPs. Prevents email spoofing |
| 5 | Why port 25 vs 587? | 25: server relay (blocked by ISPs). 587: client submission with auth |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • SMTP sends, POP3/IMAP receives                        │
## │  • IMAP > POP3 for modern multi-device use              │
## │  • Port 587 (STARTTLS) for sending, 993 for IMAPS       │
## │  • SPF + DKIM + DMARC = email authentication trifecta   │
## │  • SMTP has no built-in auth — security is add-on       │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **REST & WebSocket** — API design and real-time communication